# Eigenvalue Geometric Features

Occulus computes per-point PCA features from local neighbourhoods.  These
7 descriptors are the foundation of most learned and rule-based LiDAR classifiers:

| Feature | Formula | Interpretation |
|---|---|---|
| Linearity | (λ₁−λ₂)/λ₁ | Edges, ridge lines, power lines |
| Planarity | (λ₂−λ₃)/λ₁ | Facades, rooftops, road surfaces |
| Sphericity | λ₃/λ₁ | Vegetation, boulders, clutter |
| Omnivariance | (λ₁λ₂λ₃)^(1/3) | Overall volumetric complexity |
| Anisotropy | (λ₁−λ₃)/λ₁ | Structural elongation |
| Eigenentropy | −Σλᵢ log(λᵢ) | Local disorder / roughness |
| Curvature | λ₃/(λ₁+λ₂+λ₃) | Surface bending |

where λ₁ ≥ λ₂ ≥ λ₃ are sorted PCA eigenvalues.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from occulus.types import PointCloud
from occulus.normals import estimate_normals
from occulus.features import compute_geometric_features
from occulus.filters import voxel_downsample

## Build a Labelled Synthetic Scene

We place 4 geometric primitives with known structure:
- **Plane**: flat ground surface → high planarity
- **Edge**: sharp ridge line → high linearity
- **Sphere scatter**: noisy blob → high sphericity
- **Cylinder**: tree stem proxy → high linearity

In [ ]:
rng = np.random.default_rng(99)
parts = []
true_labels = []

# Plane (ground, 0–20 m square)
n_plane = 3000
px, py = rng.uniform(0, 20, n_plane), rng.uniform(0, 20, n_plane)
pz = rng.normal(0, 0.05, n_plane)
parts.append(np.column_stack([px, py, pz]))
true_labels.extend([0] * n_plane)  # 0 = plane

# Ridge line (linear feature)
n_line = 1000
lx = np.linspace(25, 45, n_line)
lz = rng.normal(0, 0.04, n_line)
ly = rng.normal(0, 0.08, n_line)
parts.append(np.column_stack([lx, ly, lz]))
true_labels.extend([1] * n_line)   # 1 = linear

# Spherical cluster (vegetation)
n_sphere = 1500
theta = rng.uniform(0, 2 * np.pi, n_sphere)
phi = rng.uniform(0, np.pi, n_sphere)
r = 2 + rng.normal(0, 0.15, n_sphere)
sx = 30 + r * np.sin(phi) * np.cos(theta)
sy = 10 + r * np.sin(phi) * np.sin(theta)
sz = 3 + r * np.cos(phi)
parts.append(np.column_stack([sx, sy, sz]))
true_labels.extend([2] * n_sphere)  # 2 = spherical

# Vertical cylinder (tree stem)
n_cyl = 800
angle = rng.uniform(0, 2 * np.pi, n_cyl)
cx = 40 + 0.3 * np.cos(angle) + rng.normal(0, 0.02, n_cyl)
cy = 15 + 0.3 * np.sin(angle) + rng.normal(0, 0.02, n_cyl)
cz = rng.uniform(0, 10, n_cyl)
parts.append(np.column_stack([cx, cy, cz]))
true_labels.extend([1] * n_cyl)   # also linear

xyz = np.vstack(parts).astype(np.float64)
true_labels = np.array(true_labels)
cloud = PointCloud(xyz)
print(f'Scene: {cloud.n_points:,} points — plane={n_plane}, linear={n_line+n_cyl}, sphere={n_sphere}')

## Compute Features

In [ ]:
cloud_n = estimate_normals(cloud, radius=1.0)
feats = compute_geometric_features(cloud_n, radius=1.0)

print(f'{'Feature':<15} {'Mean':>8} {'Std':>8} {'Max':>8}')
print('-' * 43)
for name, arr in [
    ('Linearity', feats.linearity),
    ('Planarity', feats.planarity),
    ('Sphericity', feats.sphericity),
    ('Omnivariance', feats.omnivariance),
    ('Anisotropy', feats.anisotropy),
    ('Eigenentropy', feats.eigenentropy),
    ('Curvature', feats.curvature),
]:
    print(f'{name:<15} {arr.mean():>8.4f} {arr.std():>8.4f} {arr.max():>8.4f}')

## Feature Maps

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
sc_kw = dict(s=2, alpha=0.8, rasterized=True)

for ax, (feat_arr, name, cmap) in zip(
    axes.flat,
    [
        (feats.linearity,    'Linearity',    'Reds'),
        (feats.planarity,    'Planarity',    'Blues'),
        (feats.sphericity,   'Sphericity',   'Greens'),
        (feats.omnivariance, 'Omnivariance', 'Oranges'),
        (feats.anisotropy,   'Anisotropy',   'Purples'),
        (feats.eigenentropy, 'Eigenentropy', 'YlOrBr'),
        (feats.curvature,    'Curvature',    'plasma'),
    ],
):
    sc = ax.scatter(xyz[:, 0], xyz[:, 1], c=feat_arr, cmap=cmap, **sc_kw)
    plt.colorbar(sc, ax=ax, shrink=0.8)
    ax.set_title(name, fontsize=10)
    ax.set_xlabel('X (m)', fontsize=8); ax.set_ylabel('Y (m)', fontsize=8)

# True labels in last panel
ax = axes.flat[-1]
label_names = {0: 'Plane', 1: 'Linear', 2: 'Spherical'}
colors = ['steelblue', 'tomato', 'forestgreen']
for lbl, col in enumerate(colors):
    mask = true_labels == lbl
    ax.scatter(xyz[mask, 0], xyz[mask, 1], s=1, c=col, alpha=0.6, label=label_names[lbl])
ax.legend(markerscale=4, fontsize=7); ax.set_title('Ground Truth', fontsize=10)
ax.set_xlabel('X (m)', fontsize=8)

plt.suptitle('Eigenvalue Geometric Features', fontsize=14)
plt.tight_layout()
plt.savefig('../../outputs/geometric_features.png', dpi=150)
plt.show()

## Simple Rule-Based Classifier

In [ ]:
# Simple threshold classifier using feature dominance
predicted = np.argmax(
    np.column_stack([feats.linearity, feats.planarity, feats.sphericity]),
    axis=1,
)
# Remap: argmax 0=linear, 1=planar, 2=spherical
# Our true labels: 0=plane, 1=linear, 2=spherical
predicted_mapped = np.where(predicted == 0, 1, np.where(predicted == 1, 0, 2))

accuracy = (predicted_mapped == true_labels).mean()
print(f'Rule-based classification accuracy: {accuracy:.2%}')
print('(No training data — purely geometric)')

**Next**: `09_international_data_guide.ipynb`